In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Paths
root = Path("../data/processed")

train = pd.read_csv(root / "train_labels.csv")
validation = pd.read_csv(root / "validation_labels.csv")

products = [
    "B00O4OR4GQ",
    "B013UJS35G",
    "B0159A0LH2",
    "B019D9HESO",
    "B01B1GE6KG",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

# Pretrained CNN input transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class BinDataset(Dataset):
    def __init__(self, dataframe, split, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = root / split
        self.transform = transform
        self.label_columns = [f"qty_{p}" for p in products]

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = self.image_dir / row["filename"]

        with Image.open(image_path) as img:
            image = self.transform(img.convert("RGB"))

        # Predict product presence, not quantity, in this first CNN experiment
        labels = (
            row[self.label_columns].to_numpy(dtype=np.float32) > 0
        ).astype(np.float32)

        return image, torch.tensor(labels)

train_dataset = BinDataset(train, "train", transform)
val_dataset = BinDataset(validation, "validation", transform)

train_loader = DataLoader(
    train_dataset, batch_size=16, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False, num_workers=0
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Number of product targets:", len(products))
print("CNN data loaders ready.")

Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Training images: 1400
Validation images: 200
Number of product targets: 5
CNN data loaders ready.


In [2]:
# Load pretrained ResNet-18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Freeze pretrained layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer to predict our 5 catalogue products
model.fc = nn.Linear(model.fc.in_features, len(products))

# Move model to GPU
model = model.to(device)

# Configure training
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

print("Pretrained ResNet-18 loaded successfully.")
print("Device:", next(model.parameters()).device)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

3.9%

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\ADMIN/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


Pretrained ResNet-18 loaded successfully.
Device: cuda:0
Trainable parameters: 2565


In [3]:
from sklearn.metrics import f1_score
import numpy as np
import torch

num_epochs = 5
best_val_f1 = -1
best_model_path = Path("../models/resnet18_best.pth")
best_model_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)

            probabilities = torch.sigmoid(logits)
            predictions = (probabilities >= 0.5).float()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predictions.cpu().numpy())

    val_loss /= len(val_dataset)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    # F1 for present products only (positive class)
    positive_f1 = f1_score(
        all_labels, all_preds, average="macro", zero_division=0
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f} | "
        f"Validation Macro F1: {positive_f1:.4f}"
    )

    # Save the best model based on validation F1
    if positive_f1 > best_val_f1:
        best_val_f1 = positive_f1
        torch.save(model.state_dict(), best_model_path)
        print("  Saved best model.")

print("\nTraining complete.")
print("Best validation Macro F1:", round(best_val_f1, 4))
print("Best model saved to:", best_model_path)

Epoch 1/5 | Train Loss: 0.3902 | Validation Loss: 0.3320 | Validation Macro F1: 0.1170
  Saved best model.
Epoch 2/5 | Train Loss: 0.3168 | Validation Loss: 0.3010 | Validation Macro F1: 0.2620
  Saved best model.
Epoch 3/5 | Train Loss: 0.2815 | Validation Loss: 0.2833 | Validation Macro F1: 0.2970
  Saved best model.
Epoch 4/5 | Train Loss: 0.2615 | Validation Loss: 0.2574 | Validation Macro F1: 0.4133
  Saved best model.
Epoch 5/5 | Train Loss: 0.2470 | Validation Loss: 0.2489 | Validation Macro F1: 0.4239
  Saved best model.

Training complete.
Best validation Macro F1: 0.4239
Best model saved to: ..\models\resnet18_best.pth


In [4]:
# Load the best model from the first training stage
model.load_state_dict(torch.load(best_model_path, map_location=device))

# Unfreeze ResNet-18's final residual block
for param in model.layer4.parameters():
    param.requires_grad = True

# Keep the final classification layer trainable
for param in model.fc.parameters():
    param.requires_grad = True

# Use a smaller learning rate for fine-tuning
optimizer = torch.optim.Adam([
    {"params": model.layer4.parameters(), "lr": 0.0001},
    {"params": model.fc.parameters(), "lr": 0.0005}
])

print("Fine-tuning setup complete.")
print(
    "Trainable parameter count:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

Fine-tuning setup complete.
Trainable parameter count: 8396293


In [5]:
num_epochs = 5
best_val_f1 = -1
fine_tuned_path = Path("../models/resnet18_finetuned_best.pth")

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_dataset)

    model.eval()
    val_loss = 0.0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)

            predictions = (torch.sigmoid(logits) >= 0.5).float()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predictions.cpu().numpy())

    val_loss /= len(val_dataset)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    val_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    print(
        f"Fine-tune Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f} | "
        f"Validation Macro F1: {val_f1:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), fine_tuned_path)
        print("  Saved best fine-tuned model.")

print("\nFine-tuning complete.")
print("Best fine-tuned validation Macro F1:", round(best_val_f1, 4))
print("Saved to:", fine_tuned_path)

Fine-tune Epoch 1/5 | Train Loss: 0.1957 | Validation Loss: 0.1261 | Validation Macro F1: 0.7871
  Saved best fine-tuned model.
Fine-tune Epoch 2/5 | Train Loss: 0.0550 | Validation Loss: 0.1129 | Validation Macro F1: 0.8546
  Saved best fine-tuned model.
Fine-tune Epoch 3/5 | Train Loss: 0.0260 | Validation Loss: 0.1019 | Validation Macro F1: 0.8543
Fine-tune Epoch 4/5 | Train Loss: 0.0138 | Validation Loss: 0.1102 | Validation Macro F1: 0.8316
Fine-tune Epoch 5/5 | Train Loss: 0.0092 | Validation Loss: 0.1036 | Validation Macro F1: 0.8601
  Saved best fine-tuned model.

Fine-tuning complete.
Best fine-tuned validation Macro F1: 0.8601
Saved to: ..\models\resnet18_finetuned_best.pth


In [6]:
from sklearn.metrics import classification_report, f1_score
import numpy as np
import torch

# Load best fine-tuned checkpoint
model.load_state_dict(
    torch.load(fine_tuned_path, map_location=device)
)
model.eval()

all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        logits = model(images)
        probs = torch.sigmoid(logits)

        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels).astype(int)
all_probs = np.array(all_probs)

# Standard threshold for product presence
all_preds = (all_probs >= 0.5).astype(int)

for i, product in enumerate(products):
    print(f"\nProduct: {product}")
    print(
        classification_report(
            all_labels[:, i],
            all_preds[:, i],
            labels=[0, 1],
            target_names=["Absent", "Present"],
            zero_division=0,
            digits=3
        )
    )

print(
    "Overall macro F1:",
    round(f1_score(
        all_labels, all_preds,
        average="macro",
        zero_division=0
    ), 4)
)


Product: B00O4OR4GQ
              precision    recall  f1-score   support

      Absent      0.972     1.000     0.986       175
     Present      1.000     0.800     0.889        25

    accuracy                          0.975       200
   macro avg      0.986     0.900     0.937       200
weighted avg      0.976     0.975     0.974       200


Product: B013UJS35G
              precision    recall  f1-score   support

      Absent      0.994     0.987     0.990       155
     Present      0.957     0.978     0.967        45

    accuracy                          0.985       200
   macro avg      0.975     0.982     0.979       200
weighted avg      0.985     0.985     0.985       200


Product: B0159A0LH2
              precision    recall  f1-score   support

      Absent      0.948     0.982     0.965       167
     Present      0.889     0.727     0.800        33

    accuracy                          0.940       200
   macro avg      0.918     0.855     0.882       200
weighted av

In [7]:
from src.order_validation import validate_order

# Example 1: Correct order
order = {"B013UJS35G": 1, "B019D9HESO": 2}
predicted = {"B013UJS35G": 1, "B019D9HESO": 2}

print(validate_order(order, predicted))

# Example 2: Incorrect quantity
order = {"B013UJS35G": 1, "B019D9HESO": 2}
predicted = {"B013UJS35G": 1, "B019D9HESO": 1}

print(validate_order(order, predicted))

ModuleNotFoundError: No module named 'src'